In [1]:
#TASK 1

from collections import Counter
import re

corpus = """
the student is studying
the student is reading
the student is writing
the teacher is teaching
the teacher is reading
the student likes python
the student likes programming
"""

# Convert corpus into words
words = re.findall(r'\b\w+\b', corpus.lower())

# Count unigram
unigram = Counter(words)

# Count bigram
bigram = Counter()

for i in range(len(words) - 1):
    bigram[(words[i], words[i + 1])] += 1

# Count trigram
trigram = Counter()

for i in range(len(words) - 2):
    trigram[(words[i], words[i + 1], words[i + 2])] += 1


# Unigram probability
def unigram_probability(word):
    return unigram[word] / len(words)


# Bigram probability
def bigram_probability(w1, w2):
    if unigram[w1] == 0:
        return 0

    return bigram[(w1, w2)] / unigram[w1]


# Trigram probability
def trigram_probability(w1, w2, w3):
    if bigram[(w1, w2)] == 0:
        return 0

    return trigram[(w1, w2, w3)] / bigram[(w1, w2)]


print("N-GRAM LANGUAGE MODEL")
print("---------------------")

n = int(input("Enter N (1, 2 or 3): "))

if n == 1:

    print("\nUNIGRAM COUNTS")

    for word, count in unigram.items():
        print(word, "=", count)

elif n == 2:

    print("\nBIGRAM COUNTS")

    for pair, count in bigram.items():
        print(pair, "=", count)

elif n == 3:

    print("\nTRIGRAM COUNTS")

    for triple, count in trigram.items():
        print(triple, "=", count)

else:
    print("Invalid N")
    exit()


# Next word prediction
sentence = input("\nEnter incomplete sentence: ")
input_words = sentence.lower().split()

predictions = []

if n == 1:

    for word in unigram:
        probability = unigram_probability(word)
        predictions.append((word, probability))

elif n == 2:

    last_word = input_words[-1]

    for word in unigram:

        probability = bigram_probability(last_word, word)

        if probability > 0:
            predictions.append((word, probability))

elif n == 3:

    if len(input_words) < 2:
        print("Enter at least two words")
        exit()

    w1 = input_words[-2]
    w2 = input_words[-1]

    for word in unigram:

        probability = trigram_probability(w1, w2, word)

        if probability > 0:
            predictions.append((word, probability))


predictions.sort(key=lambda x: x[1], reverse=True)

print("\nTOP 5 NEXT WORDS")

for word, probability in predictions[:5]:

    print(word, "Probability =", round(probability, 3))


# Demonstrate unseen N-gram
print("\nUNSEEN N-GRAM")

print(
    "Probability of 'student football' =",
    bigram_probability("student", "football")
)

N-GRAM LANGUAGE MODEL
---------------------
Enter N (1, 2 or 3): 2

BIGRAM COUNTS
('the', 'student') = 5
('student', 'is') = 3
('is', 'studying') = 1
('studying', 'the') = 1
('is', 'reading') = 2
('reading', 'the') = 2
('is', 'writing') = 1
('writing', 'the') = 1
('the', 'teacher') = 2
('teacher', 'is') = 2
('is', 'teaching') = 1
('teaching', 'the') = 1
('student', 'likes') = 2
('likes', 'python') = 1
('python', 'the') = 1
('likes', 'programming') = 1

Enter incomplete sentence: the student

TOP 5 NEXT WORDS
is Probability = 0.6
likes Probability = 0.4

UNSEEN N-GRAM
Probability of 'student football' = 0.0


In [2]:
#TASK 2

from collections import Counter

corpus = """
the student is studying
the student is reading
the student is writing
the teacher is teaching
the teacher is reading
the student likes python
the teacher likes python
"""

words = corpus.lower().split()

unigram = Counter(words)
bigram = Counter()
trigram = Counter()

# Count bigrams and trigrams
for i in range(len(words) - 1):
    bigram[(words[i], words[i + 1])] += 1

for i in range(len(words) - 2):
    trigram[(words[i], words[i + 1], words[i + 2])] += 1


# Unigram probability
def unigram_probability(word):
    return unigram[word] / len(words)


# Bigram probability
def bigram_probability(w1, w2):

    if unigram[w1] == 0:
        return 0

    return bigram[(w1, w2)] / unigram[w1]


# Trigram probability
def trigram_probability(w1, w2, w3):

    if bigram[(w1, w2)] == 0:
        return 0

    return trigram[(w1, w2, w3)] / bigram[(w1, w2)]


# Backoff probability
def backoff(w1, w2, word):

    p = trigram_probability(w1, w2, word)

    if p > 0:
        return p

    p = bigram_probability(w2, word)

    if p > 0:
        return p

    return unigram_probability(word)


# Deleted interpolation
def interpolation(w1, w2, word):

    p1 = unigram_probability(word)
    p2 = bigram_probability(w2, word)
    p3 = trigram_probability(w1, w2, word)

    # Weights
    lambda1 = 0.2
    lambda2 = 0.3
    lambda3 = 0.5

    return (
        lambda1 * p1 +
        lambda2 * p2 +
        lambda3 * p3
    )


def predict(sentence, method):

    words_input = sentence.lower().split()

    if len(words_input) < 2:
        print("Enter at least two words")
        return

    w1 = words_input[-2]
    w2 = words_input[-1]

    results = []

    for word in unigram:

        if method == "unsmoothed":

            probability = trigram_probability(
                w1, w2, word
            )

        elif method == "backoff":

            probability = backoff(
                w1, w2, word
            )

        else:

            probability = interpolation(
                w1, w2, word
            )

        if probability > 0:
            results.append((word, probability))

    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    for word, probability in results[:5]:

        print(
            word,
            "Probability =",
            round(probability, 3)
        )


print("LANGUAGE PREDICTION SYSTEM")
print("--------------------------")

sentence = input("Enter sentence: ")

print("\nUNSMOOTHED MODEL")
predict(sentence, "unsmoothed")

print("\nBACKOFF MODEL")
predict(sentence, "backoff")

print("\nDELETED INTERPOLATION")
predict(sentence, "interpolation")

LANGUAGE PREDICTION SYSTEM
--------------------------
Enter sentence: teacher is

UNSMOOTHED MODEL
reading Probability = 0.5
teaching Probability = 0.5

BACKOFF MODEL
reading Probability = 0.5
teaching Probability = 0.5
the Probability = 0.25
studying Probability = 0.2
writing Probability = 0.2

DELETED INTERPOLATION
reading Probability = 0.384
teaching Probability = 0.317
studying Probability = 0.067
writing Probability = 0.067
the Probability = 0.05


In [3]:
#TASK 3

from collections import Counter
import math

training_text = """
the student is studying
the student is reading
the student is writing
the teacher is teaching
the teacher is reading
"""

test_text = """
the student is reading
the teacher is teaching
"""

train_words = training_text.lower().split()
test_words = test_text.lower().split()

# Unigram
unigram = Counter(train_words)

# Bigram
bigram = Counter()

for i in range(len(train_words) - 1):
    bigram[(train_words[i], train_words[i + 1])] += 1


# Unigram probability
def unigram_probability(word):

    return unigram[word] / len(train_words)


# Bigram probability
def bigram_probability(w1, w2):

    if unigram[w1] == 0:
        return 0

    return bigram[(w1, w2)] / unigram[w1]


# Calculate entropy
def calculate_entropy():

    total = 0
    count = 0

    for i in range(1, len(test_words)):

        w1 = test_words[i - 1]
        w2 = test_words[i]

        probability = bigram_probability(w1, w2)

        if probability > 0:

            total = total + (-math.log2(probability))

            count = count + 1

    if count == 0:
        return 0

    return total / count


print("N-GRAM ENTROPY")
print("--------------")

entropy = calculate_entropy()

print("Entropy =", round(entropy, 3))


# Test next-word prediction
print("\nNEXT WORD PREDICTION")

word = input("Enter a word: ")

predictions = []

for next_word in unigram:

    probability = bigram_probability(
        word,
        next_word
    )

    if probability > 0:

        predictions.append(
            (next_word, probability)
        )


predictions.sort(
    key=lambda x: x[1],
    reverse=True
)

print("\nPossible next words:")

for next_word, probability in predictions[:5]:

    print(
        next_word,
        "Probability =",
        round(probability, 3)
    )

N-GRAM ENTROPY
--------------
Entropy = 0.958

NEXT WORD PREDICTION
Enter a word: student

Possible next words:
is Probability = 1.0


In [4]:
#TASK 4
from collections import Counter

# Training data
training_sentences = [
    ("the student reads",
     ["DT", "NN", "VBZ"]),

    ("the teacher teaches",
     ["DT", "NN", "VBZ"]),

    ("she is reading",
     ["PRP", "VBZ", "VBG"]),

    ("he likes python",
     ["PRP", "VBZ", "NNP"]),

    ("the smart student writes",
     ["DT", "JJ", "NN", "VBZ"])
]


# Build word-tag dictionary
word_tags = {}

for sentence, tags in training_sentences:

    words = sentence.split()

    for word, tag in zip(words, tags):

        word_tags[word] = tag


# -------------------------------
# RULE-BASED TAGGER
# -------------------------------

def rule_based(sentence):

    words = sentence.lower().split()

    result = []

    for word in words:

        if word in ["i", "you", "he", "she", "we", "they"]:
            tag = "PRP"

        elif word in ["the", "a", "an"]:
            tag = "DT"

        elif word in ["and", "or", "but"]:
            tag = "CC"

        elif word in ["in", "on", "at", "with", "from", "to"]:
            tag = "IN"

        elif word in ["smart", "good", "big"]:
            tag = "JJ"

        elif word.endswith("ly"):
            tag = "RB"

        elif word.endswith("ing"):
            tag = "VBG"

        elif word.endswith("s"):
            tag = "VBZ"

        elif word in ["student", "teacher", "book", "python"]:
            tag = "NN"

        else:
            tag = "NN"

        result.append((word, tag))

    return result


# -------------------------------
# STOCHASTIC TAGGER
# -------------------------------

def stochastic(sentence):

    words = sentence.lower().split()

    result = []

    for word in words:

        if word in word_tags:

            tag = word_tags[word]

        else:

            if word.endswith("ing"):
                tag = "VBG"

            elif word.endswith("ly"):
                tag = "RB"

            elif word.endswith("s"):
                tag = "VBZ"

            else:
                tag = "NN"

        result.append((word, tag))

    return result


# -------------------------------
# TRANSFORMATION-BASED TAGGER
# -------------------------------

def transformation(sentence):

    result = rule_based(sentence)

    for i in range(1, len(result)):

        word, tag = result[i]

        previous_word, previous_tag = result[i - 1]

        # Rule:
        # Pronoun + word -> Verb
        if previous_tag == "PRP" and tag == "NN":
            result[i] = (word, "VB")

        # Word ending with ing -> VBG
        if word.endswith("ing"):
            result[i] = (word, "VBG")

        # Word ending with ly -> RB
        if word.endswith("ly"):
            result[i] = (word, "RB")

    return result


# -------------------------------
# MAIN PROGRAM
# -------------------------------

print("POS TAGGING SYSTEM")
print("------------------")

sentence = input(
    "Enter an English sentence: "
)


print("\nRule-Based Tagger")

result1 = rule_based(sentence)

for word, tag in result1:
    print(word, "->", tag)


print("\nStochastic Tagger")

result2 = stochastic(sentence)

for word, tag in result2:
    print(word, "->", tag)


print("\nTransformation-Based Tagger")

result3 = transformation(sentence)

for word, tag in result3:
    print(word, "->", tag)


print("\nPenn Treebank Tags")
print("NN  = Noun")
print("VB  = Verb")
print("VBZ = Verb")
print("VBG = Verb + ing")
print("JJ  = Adjective")
print("RB  = Adverb")
print("PRP = Pronoun")
print("IN  = Preposition")
print("CC  = Conjunction")
print("DT  = Determiner")

POS TAGGING SYSTEM
------------------
Enter an English sentence: She is reading

Rule-Based Tagger
she -> PRP
is -> VBZ
reading -> VBG

Stochastic Tagger
she -> PRP
is -> VBZ
reading -> VBG

Transformation-Based Tagger
she -> PRP
is -> VBZ
reading -> VBG

Penn Treebank Tags
NN  = Noun
VB  = Verb
VBZ = Verb
VBG = Verb + ing
JJ  = Adjective
RB  = Adverb
PRP = Pronoun
IN  = Preposition
CC  = Conjunction
DT  = Determiner
